## Plate Reconstruction QC

Following plate reconstruction, Smart-seq3 data and FACS index sorting metadata were combined and stored in a single `.h5ad` file to facilitate quality control and downstream analyses.


In [ ]:
import scanpy as sc
import omicverse as ov
import numpy as np
import pandas as pd
import os
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns

ov.plot_set(dpi=150,font_path='Arial')

In [ ]:
# Set the mouse and batch parameters
mouse = "PC118"
mouse_batch = "PC118-3"

# Define input path for the count matrix
input_path = f"F:/Data/snRNAseq/PlaceCell/Batch3_rawdata/PM-20250902-zhaoming-1253/cellranger/{mouse_batch}/SS3/report"

# Load the count matrix and transpose it
adata_raw = sc.read_csv(os.path.join(input_path, "count_matrix.csv")).T
adata = adata_raw.copy()

# Define the FACS index file path
facs_path = f"F:/Data/PlaceCells_snRNAseq_RawData/FACS/{mouse}/{mouse_batch}/index.csv"

# Load FACS data
FACS_index = pd.read_csv(facs_path)


In [ ]:
adata

In [ ]:
# Extract well ID (e.g., A1, B12) from cell/nucleus identifiers in obs_names
adata.obs["well"] = (
    adata.obs_names.to_series()
    .str.extract(r'([A-H]\d+)$')[0]   # Capture trailing well code from sample name
)

# Display extracted well IDs
adata.obs["well"]


In [ ]:
# import re

# def rename_index(name):
#     m = re.search(r'(\d+)$', name)
#     if m:
#         num = int(m.group(1))
#         return name[:m.start()] + str(num - 9)
#     else:
#         return name

# adata.obs_names = [rename_index(n) for n in adata.obs_names]
# adata.obs_names_make_unique()


In [ ]:
# Update label
sorting_wells = dict(zip(FACS_index["well"], FACS_index["label"]))
property = adata.obs["well"].map(sorting_wells).fillna(np.nan).tolist()
adata.obs["property"] = property

In [ ]:
adata = adata[adata.obs["property"].notna(), :]
adata

In [ ]:
cell_number_830 = (adata.obs["property"]== 830).sum()
cell_number_750 = (adata.obs["property"]== 750).sum()
print("750 labeled cells:", cell_number_750)
print("830 labeled cells:", cell_number_830)

In [ ]:
adata = ov.pp.qc(
    adata,
    mt_startswith='mt-',
    tresh={'mito_perc': 0.05,
           'nUMIs': 5000,
           'detected_genes': 500},
    min_cells=1,
    doublets=False,
)

In [ ]:
adata = adata[adata.obs["detected_genes"] <= 10000].copy()

In [ ]:
sc.pl.violin(
    adata,
    ["detected_genes", "nUMIs", "mito_perc"],
    jitter=0.4,
    multi_panel=True,
    size=5
)

In [ ]:
cell_number_830 = (adata.obs["property"]== 830).sum()
cell_number_750 = (adata.obs["property"]== 750).sum()
print("750 labeled cells after QC:", cell_number_750)
print("830 labeled cells after QC:", cell_number_830)

In [ ]:
adata.obs['property'] = adata.obs['property'].astype('category')
sc.pl.violin(
    adata,
    keys=['detected_genes','nUMIs'],
    groupby='property'
)

In [ ]:
adata.X.max()

In [ ]:
adata.write_h5ad(os.path.join(input_path, f"{mouse_batch}_counts_martix_QC.h5ad"))